# Preprocessing

Load the six raw CSVs, unify the schema, and save two processed files:

- `kb_combined.csv` — the knowledge base (MedQuAD + MedlinePlus + PubMedQA + MedRedQA)
- `eval_combined.csv` — the evaluation set (PubMedQA labeled + HealthSearchQA)

Same unified schema as before:

| Field | Description |
|-------|-------------|
| `doc_id` | unique within the file |
| `source` | medquad / medlineplus / pubmedqa / medredqa |
| `source_id` | original id from the raw source |
| `category` | qtype when available, else empty |
| `question` | the question (synthesized for MedlinePlus) |
| `context` | background text (abstract, Reddit body, etc.) |
| `answer` | the answer / summary |

Second pass over the raw data — the first version had HTML tags leaking into text, kept site boilerplate in MedQuAD answers, and shipped HealthSearchQA with duplicates and non-medical queries.

In [1]:
import re
import ast
import html
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup

In [2]:
ROOT = Path.cwd().parent
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(sorted(f.name for f in RAW_DIR.glob("*.csv")))

['healthsearchqa_raw.csv', 'medlineplus_raw.csv', 'medquad_raw.csv', 'medredqa_raw.csv', 'pubmedqa_eval_raw.csv', 'pubmedqa_kb_raw.csv']


## Helpers

### Text cleaning

In [3]:
# Only strip real HTML tags. The old <[^>]+> pattern was matching things
# like "a < b > c" or "<5mg" in clinical text.
TAG_RE = re.compile(r"</?[a-zA-Z][^>]*>")


def clean_text(t):
    if t is None or (isinstance(t, float) and pd.isna(t)):
        return ""
    t = str(t)
    t = TAG_RE.sub(" ", t)
    t = re.sub(r"\s+", " ", t)
    t = re.sub(r"\s+\?", "?", t)
    t = re.sub(r"\?+", "?", t)
    t = t.replace(" .", ".").replace(" ,", ",")
    return t.strip()


def html_to_text(x):
    # MedlinePlus summaries are real HTML (lists, links, headings).
    # Regex alone isn't enough, need a parser.
    if pd.isna(x):
        return ""
    x = str(x)
    x = BeautifulSoup(x, "html.parser").get_text(" ")
    x = html.unescape(x)
    x = re.sub(r"\s+", " ", x)
    return x.strip()

In [4]:
# MedQuAD answers were scraped from NIH sites, so they carry a lot of
# site chrome at the tail. Cut the answer the moment we hit a marker.
BOILERPLATE_MARKERS = [
    "More on:",
    "Top of Page",
    "Related Pages",
    "Images and logos on this website",
    "This graphic notice means",
    "Go to:",
    "Back to Top",
]


def strip_boilerplate(t):
    if not t:
        return ""
    for m in BOILERPLATE_MARKERS:
        idx = t.find(m)
        if idx != -1:
            t = t[:idx]
    return re.sub(r"\s+", " ", t).strip()

In [5]:
def parse_context(x):
    if isinstance(x, dict):
        return " ".join(str(c) for c in x.get("contexts", []))

    if isinstance(x, str):
        # Only try literal_eval when the string actually looks like a dict.
        # This is what was firing the SyntaxWarnings before.
        if x.strip().startswith("{"):
            try:
                d = ast.literal_eval(x)
                if isinstance(d, dict) and "contexts" in d:
                    return " ".join(str(c) for c in d["contexts"])
            except (ValueError, SyntaxError):
                pass

        # Fallback for the stringified numpy array form
        m = re.search(r"'contexts':\s*array\(\[(.*?)\]", x, re.DOTALL)
        if m:
            parts = re.findall(r"'((?:[^'\\]|\\.)*)'", m.group(1))
            if parts:
                return " ".join(parts)

    return str(x)

In [6]:
# HealthSearchQA is scraped from Google autocomplete, so it also catches
# non-health queries. Drop the obvious ones.
NON_MEDICAL_RE = re.compile(
    r"\b(movie|film|actor|actress|song|album|recipe|bitcoin|crypto|"
    r"javascript|python|minecraft|fortnite|football|basketball)\b",
    re.IGNORECASE,
)


def looks_medical(q):
    if not isinstance(q, str) or not q.strip():
        return False
    return not NON_MEDICAL_RE.search(q)

## 1. MedQuAD

Clean text, strip boilerplate, drop the `What is (are) ?` template rows,
then dedup on question keeping the longest answer.

In [7]:
raw = pd.read_csv(RAW_DIR / "medquad_raw.csv")
print("raw:", len(raw))

mq = pd.DataFrame({
    "source": "medquad",
    "source_id": raw.index.astype(str),
    "category": raw["qtype"].fillna("").astype(str),
    "question": raw["Question"],
    "context": "",
    "answer": raw["Answer"],
})

for col in ["question", "answer"]:
    mq[col] = mq[col].apply(clean_text)

mq["answer"] = mq["answer"].apply(strip_boilerplate)

# drop malformed / empty questions
mq = mq[mq["question"].str.len() > 5]
mq = mq[~mq["question"].str.contains(r"\(are\)\s*\?", regex=True, na=False)]

# keep the longest answer per question
mq["_len"] = mq["answer"].str.len()
mq = mq.sort_values("_len", ascending=False)
mq = mq.drop_duplicates(subset=["question"]).drop(columns="_len")
mq = mq.reset_index(drop=True)

print("after clean:", len(mq))
print("dropped:", len(raw) - len(mq))

raw: 16407
after clean: 14978
dropped: 1429


## 2. MedlinePlus

No question field — we synthesize one from the title.
The summary column needs a real HTML parser, not regex.

In [8]:
raw = pd.read_csv(RAW_DIR / "medlineplus_raw.csv")
print("raw:", len(raw))

ml = pd.DataFrame({
    "source": "medlineplus",
    "source_id": raw["url"],
    "category": "",
    "question": "What is " + raw["title"].astype(str) + "?",
    "context": "",
    "answer": raw["summary"].apply(html_to_text),
})

ml["question"] = ml["question"].apply(clean_text)
ml = ml[ml["answer"].str.len() > 0]
ml = ml.drop_duplicates(subset=["source_id"]).reset_index(drop=True)

print("after clean:", len(ml))

raw: 1017
after clean: 1015


## 3. PubMedQA (KB)

The `context` field is the PubMed abstract. Apply `parse_context` to
unwrap the stringified numpy array. Drop rows where parsing failed.

In [9]:
raw = pd.read_csv(RAW_DIR / "pubmedqa_kb_raw.csv")
print("raw:", len(raw))

pq = pd.DataFrame({
    "source": "pubmedqa",
    "source_id": raw["pubid"].astype(str),
    "category": "",
    "question": raw["question"],
    "context": raw["context"].apply(parse_context),
    "answer": raw["long_answer"],
})

for col in ["question", "context", "answer"]:
    pq[col] = pq[col].apply(clean_text)

# keep only rows with real content
pq = pq[(pq["context"].str.len() > 20) & (pq["question"].str.len() > 5)]
pq = pq.drop_duplicates(subset=["source_id"]).reset_index(drop=True)

print("after clean:", len(pq))

raw: 211269
after clean: 211267


## 4. MedRedQA

Question = title, context = user's description, answer = physician response.
Drop very short answers — they're usually casual replies with no content.

In [10]:
raw = pd.read_csv(RAW_DIR / "medredqa_raw.csv")
print("raw:", len(raw))

mr = pd.DataFrame({
    "source": "medredqa",
    "source_id": raw["id"].astype(str),
    "category": "",
    "question": raw["title"],
    "context": raw["body"],
    "answer": raw["response"],
})

for col in ["question", "context", "answer"]:
    mr[col] = mr[col].apply(clean_text)

print("before filter:", len(mr))
mr = mr[mr["answer"].str.len() >= 100]
mr = mr[mr["question"].str.len() > 5]
mr = mr.drop_duplicates(subset=["source_id"]).reset_index(drop=True)
print("after filter:", len(mr))

raw: 40602
before filter: 40602
after filter: 29581


## 5. Combine KB

In [11]:
kb = pd.concat([mq, ml, pq, mr], ignore_index=True)

# keep rows that have at least an answer or a context
kb = kb[(kb["answer"] != "") | (kb["context"] != "")].reset_index(drop=True)
kb.insert(0, "doc_id", kb.index.astype(str))

print("total:", len(kb))
print(kb["source"].value_counts())

total: 256841
source
pubmedqa       211267
medredqa        29581
medquad         14978
medlineplus      1015
Name: count, dtype: int64


In [12]:
kb.head(3)

,doc_id,source,source_id,category,question,context,answer
0,0,medquad,3005,treatment,What are the treatments for Breast Cancer?,,Key Points - There are different types of trea...
1,1,medquad,3037,treatment,What are the treatments for Childhood Soft Tis...,,Key Points - There are different types of trea...
2,2,medquad,2827,information,What is (are) Childhood Vascular Tumors?,,Key Points - Childhood vascular tumors form fr...


## 6. Eval set

PubMedQA labeled + HealthSearchQA. HealthSearchQA needs a bit more work:
the `id` column comes as float and there are trailing empty rows.

In [13]:
raw = pd.read_csv(RAW_DIR / "pubmedqa_eval_raw.csv")

pq_eval = pd.DataFrame({
    "source": "pubmedqa_eval",
    "source_id": raw["pubid"].astype(str),
    "question": raw["question"],
    "context": raw["context"].apply(parse_context),
    "answer": raw["long_answer"],
})

for col in ["question", "context", "answer"]:
    pq_eval[col] = pq_eval[col].apply(clean_text)

pq_eval = pq_eval[pq_eval["question"].str.len() > 5]
print("pubmedqa_eval:", len(pq_eval))

pubmedqa_eval: 1000


In [14]:
raw = pd.read_csv(RAW_DIR / "healthsearchqa_raw.csv")
print("raw:", len(raw))

# drop trailing empty rows, then fix the id type
raw = raw.dropna(subset=["id", "question"]).copy()
raw["id"] = raw["id"].astype(int)

# dedup and filter non-medical queries
raw = raw.drop_duplicates(subset=["question"])
before = len(raw)
raw = raw[raw["question"].apply(looks_medical)]
print(f"dropped {before - len(raw)} non-medical queries")

hs = pd.DataFrame({
    "source": "healthsearchqa",
    "source_id": raw["id"].astype(str),
    "question": raw["question"],
    "context": "",
    "answer": "",
})

hs["question"] = hs["question"].apply(clean_text)
hs = hs[hs["question"] != ""].reset_index(drop=True)
print("after clean:", len(hs))

raw: 4436
dropped 1 non-medical queries
after clean: 3155


In [15]:
eval_set = pd.concat([pq_eval, hs], ignore_index=True)
eval_set = eval_set[eval_set["question"] != ""].reset_index(drop=True)
eval_set.insert(0, "eval_id", eval_set.index.astype(str))

print("total:", len(eval_set))
print(eval_set["source"].value_counts())

total: 4155
source
healthsearchqa    3155
pubmedqa_eval     1000
Name: count, dtype: int64


## 7. Save

In [16]:
kb.to_csv(PROCESSED_DIR / "kb_combined.csv", index=False)
eval_set.to_csv(PROCESSED_DIR / "eval_combined.csv", index=False)

print(f"kb_combined.csv   : {len(kb):,} rows")
print(f"eval_combined.csv : {len(eval_set):,} rows")

kb_combined.csv   : 256,841 rows
eval_combined.csv : 4,155 rows


In [17]:
summary = (
    kb.groupby("source")
    .agg(
        rows=("doc_id", "size"),
        has_context=("context", lambda s: round((s != "").mean(), 3)),
        has_answer=("answer", lambda s: round((s != "").mean(), 3)),
    )
)
print(summary)

               rows  has_context  has_answer
source                                      
medlineplus    1015          0.0         1.0
medquad       14978          0.0         1.0
medredqa      29581          1.0         1.0
pubmedqa     211267          1.0         1.0


## 8. What changed vs the first pass

| Issue | Fix |
|-------|-----|
| HTML tags leaking into text | `TAG_RE = </?[a-zA-Z][^>]*>` instead of `<[^>]+>` |
| MedlinePlus HTML | BeautifulSoup instead of regex |
| PubMedQA `SyntaxWarning` | Only run `literal_eval` when string starts with `{` |
| MedQuAD site chrome | `strip_boilerplate` cuts at known markers |
| MedQuAD dup + empty rows | Dedup on question, keep longest answer |
| HealthSearchQA id float | `dropna` then `astype(int)` |
| HealthSearchQA dups | `drop_duplicates(subset="question")` |
| HealthSearchQA non-medical | `looks_medical` regex filter |
| KB / eval dedup | Per-source dedup on `source_id` (or question) |

Next: `03_category_labelling.ipynb`